# 6장
## 6.4 Word2Vec
* 임베딩 클래스

```
import torch

embedding = torch.nn.Embedding(
    num_embeddings,
    embedding_dim,
    padding_idx = None,
    max_norm = None,
    norm_type = 2.0
)
```

- 기본 Skip-gram 클래스

In [ ]:
from torch import nn

class VanillaSkipgram(nn.Module):
  def __init__(self, vocab_size, embedding_dim):
    super().__init__()
    self.embedding = nn.Embedding(
        num_embeddings = vocab_size,
        embedding_dim = embedding_dim
    )
    self.linear = nn.Linear(
        in_features = embedding_dim,
        out_features = vocab_size
    )

  def forward(self, input_ids):
    embeddings = self.embedding(input_ids)
    output = self.linear(embeddings)
    return output

* 영화 리뷰 데이터세트 전처리

In [ ]:
!pip install Korpora konlpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 86.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [konlpy]


In [ ]:
import pandas as pd
from Korpora import Korpora
from konlpy.tag import Okt

corpus = Korpora.load("nsmc")
corpus = pd.DataFrame(corpus.test)

tokenizer = Okt()
tokens = [tokenizer.morphs(review) for review in corpus.text]
print(tokens[:3])


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/



[nsmc] download ratings_train.txt: 14.6MB [00:00, 58.0MB/s]                            
[nsmc] download ratings_test.txt: 4.90MB [00:00, 17.6MB/s]                           


[['굳', 'ㅋ'], ['GDNTOPCLASSINTHECLUB'], ['뭐', '야', '이', '평점', '들', '은', '....', '나쁘진', '않지만', '10', '점', '짜', '리', '는', '더', '더욱', '아니잖아']]


* 단어 사전 구축

In [ ]:
from collections import Counter

def build_vocab(corpus, n_vocab, special_tokens):
  counter = Counter()
  for tokens in corpus:
    counter.update(tokens)
  vocab = special_tokens
  for token, count in counter.most_common(n_vocab):
    vocab.append(token)
  return vocab

vocab = build_vocab(corpus = tokens, n_vocab = 5000, special_tokens=["<unk>"])
token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx : token for idx, token in enumerate(vocab)}

print(vocab[:10])
print(len(vocab))

['<unk>', '.', '이', '영화', '의', '..', '가', '에', '...', '을']
5001


* Skip-gram의 단어 쌍 추출

In [ ]:
def get_word_pairs(tokens, window_size):
  pairs = []
  for sentence in tokens:
    sentence_length = len(sentence)
    for idx, center_word in enumerate(sentence):
      window_start = max(0, idx - window_size)
      window_end = min(sentence_length, idx + window_size + 1)
      center_word = sentence[idx]
      context_words = sentence[window_start:idx] + sentence[idx+1:window_end]
      for context_word in context_words:
        pairs.append([center_word, context_word])
    return pairs

word_pairs = get_word_pairs(tokens, window_size = 2)
print(word_pairs[:5])

[['굳', 'ㅋ'], ['ㅋ', '굳']]


* 인덱스 쌍 변환

In [ ]:
def get_index_pairs(word_pairs, token_to_id):
  pairs = []
  unk_index = token_to_id["<unk>"]
  for word_pair in word_pairs:
    center_word, context_word = word_pair
    center_index = token_to_id.get(center_word, unk_index)
    context_index = token_to_id.get(context_word, unk_index)
    pairs.append([center_index, context_index])
  return pairs

index_pairs = get_index_pairs(word_pairs, token_to_id)
print(index_pairs[:5])

[[595, 100], [100, 595]]


* 데이터로더 적용

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

index_pairs = torch.tensor(index_pairs)
center_indexs = index_pairs[:, 0]
context_indexs = index_pairs[:, 1]

dataset = TensorDataset(center_indexs, context_indexs)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

/tmp/ipykernel_752/2444223511.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  index_pairs = torch.tensor(index_pairs)


* Skip-gram 모델 준비 작업

In [ ]:
from torch import optim

device = "cuda" if torch.cuda.is_available() else "cpu"
word2vec = VanillaSkipgram(vocab_size = len(token_to_id), embedding_dim = 128).to(device)
criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.SGD(word2vec.parameters(), lr = 0.1)

* 모델 학습

In [ ]:
for epoch in range(10):
  cost = 0.0
  for input_ids, target_ids in dataloader:
    input_ids = input_ids.to(device)
    target_ids = target_ids.to(device)

    logits = word2vec(input_ids)
    loss = criterion(logits, target_ids)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    cost += loss

  cost = cost / len(dataloader)
  print(f"Epoch: {epoch+1:4d}, Cost : {cost:3f}")

Epoch:    1, Cost : 9.686184
Epoch:    2, Cost : 3.591859
Epoch:    3, Cost : 0.096552
Epoch:    4, Cost : 0.052555
Epoch:    5, Cost : 0.038061
Epoch:    6, Cost : 0.030155
Epoch:    7, Cost : 0.025076
Epoch:    8, Cost : 0.021507
Epoch:    9, Cost : 0.018851
Epoch:   10, Cost : 0.016792


* 임베딩 값 추출

In [ ]:
token_to_embedding = dict()
embedding_matrix = word2vec.embedding.weight.detach().cpu().numpy()

for word, embedding in zip(vocab, embedding_matrix):
  token_to_embedding[word] = embedding

index = 30
token = vocab[30]
token_embedding = token_to_embedding[token]
print(token)
print(token_embedding)

연기
[ 0.1551197  -0.65172005  0.25621232  1.7234108  -1.8010459   0.78311944
  0.784888    1.3398051  -0.23883219  0.9310138   1.7384559  -1.3111002
  0.2706012   1.5881333   0.4775739   0.2780127   1.9101515  -0.11945815
 -0.6854068   1.9509414  -0.28459817 -2.7477124   0.41235644 -0.4416673
 -1.2346939  -0.05393753 -0.7618459  -0.81329805  0.5975488   1.6059024
 -2.0282784  -0.7312231  -0.3052912   0.6338141  -0.47557893 -0.11770225
 -1.18973    -0.64293516 -0.38225043  1.0481538  -0.12524842  0.9790465
  2.6518416   1.2548157  -1.0069615   0.27817744 -1.6437095   1.6211709
 -0.26477513  1.0952466  -1.1524295  -0.27739418  0.7366484  -0.18071511
  0.9830007   0.22795981  0.17526434 -0.7263239   0.2802423  -0.60496384
  0.4204828   0.23240691  0.11578336 -0.6254004   0.9462354  -0.7850228
 -2.7697766  -0.527391   -1.5249546  -0.13612334 -0.77025056  1.1267922
  1.0767604   0.9596886  -0.37499243 -1.0285469   0.9026349  -1.5737612
 -1.1497236  -0.09221525 -0.86112225 -0.24905759 -0.5232

* 단어 임베딩 유사도 계산

In [ ]:
import numpy as np
from numpy.linalg import norm

def cosine_similarity(a, b):
  cosine = np.dot(b, a) / (norm(b, axis = 1) * norm(a))
  return cosine

def top_n_index(cosine_matrix, n):
  closest_indexes = cosine_matrix.argsort()[::1]
  top_n = closest_indexes[1:n+1]
  return top_n

cosine_matrix = cosine_similarity(token_embedding, embedding_matrix)
top_n = top_n_index(cosine_matrix, n = 5)

print(f"{token}와 가장 유사한 5 개 단어")
for index in top_n:
  print(f"{id_to_token[index]} - 유사도 : {cosine_matrix[index]:.4f}")


연기와 가장 유사한 5 개 단어
주가 - 유사도 : -0.2925
록키 - 유사도 : -0.2901
졸 - 유사도 : -0.2802
ㅎ - 유사도 : -0.2738
년도 - 유사도 : -0.2674


## 모델 실습: Gensim

In [ ]:
#젠심 라이브러리 설치
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 55.1 MB/s  0:00:00


* Word2Vec 클래스

```
word2vec = gensim.models.Word2Vec(
    sentences = None,
    corpus_file = None,
    vector_size = 100,
    alpha = 0.025,
    window = 5,
    min_count = 5,
    workers = 3,
    sg = 0,
    hs = 0,
    cbow_mean = 1,
    negative = 5,
    ns_exponent = 0.75,
    max_final_vocab = None,
    epochs = 5,
    batch_words = 10000
)
```

* Word2Vec 모델 학습

In [ ]:
from gensim.models import Word2Vec

word2vec = Word2Vec(
    sentences = tokens,
    vector_size = 128,
    window = 5,
    min_count = 1,
    sg = 1,
    epochs = 3,
    max_final_vocab = 10000
)

# word2vec.save("../models/word2vec.model")
# word2vec = Word2Vec.load("../models/word2vec.model")

* 임베딩 추출 및 유사도 계산

In [ ]:
word = "연기"
print(word2vec.wv[word])
print(word2vec.wv.most_similar(word, topn=5))
print(word2vec.wv.similarity(w1=word, w2 = "연기력"))

[-2.85456598e-01 -2.63555169e-01  1.20392814e-01  6.30600333e-01
  8.22185576e-02  3.58849764e-02 -1.02224343e-01  2.01369166e-01
 -3.72762948e-01  1.76576391e-01  2.25322306e-01 -1.83925077e-01
 -3.47616225e-02  3.03083599e-01 -3.46922189e-01  5.59036247e-02
 -2.05041975e-01  2.88011104e-01 -3.02891612e-01 -2.98854560e-02
  5.07760406e-01  1.51627108e-01 -1.71011183e-02 -2.95250416e-02
 -2.64783800e-02 -1.62534975e-02 -4.60473210e-01  2.57516026e-01
  3.57002556e-01 -7.76253790e-02 -4.65970010e-01  2.58155704e-01
  5.06376386e-01  1.40242949e-01 -9.66325998e-02 -6.34481966e-01
  2.38226175e-01 -2.15456754e-01  1.11331400e-02 -3.65208119e-01
 -3.16245817e-02  3.88728887e-01 -7.55422842e-03 -3.74765337e-01
 -3.99608046e-01  1.84648603e-01 -2.14286044e-01 -2.31162712e-01
 -2.23501842e-03 -3.84249650e-02  6.43028975e-01  1.84049562e-01
  1.16081096e-01  1.23499654e-01 -2.26999491e-01 -5.37920110e-02
  1.60350829e-01  1.07114445e-02 -2.69097388e-01 -1.61095336e-02
  2.83908043e-02 -2.45731

## 6.5 FastText
* FastText 클래스


```
fasttext = gensim.models.FastText(
    sentences = None,
    corpus_file = None,
    vector_size = 100,
    alpha = 0.025,
    window = 5,
    min_count = 5,
    workers = 3,
    sg = 0,
    hs = 0,
    cbow_mean = 1,
    negative = 5,
    ns_exponent = 0.75,
    max_final_voacb = None,
    epochs = 5,
    batch_words = 10000,
    min_n = 3,
    max_n = 6
)
```

* KorNLI 데이터세트 전처리

In [ ]:
from Korpora import Korpora

corpus = Korpora.load("kornli")
corpus_texts = corpus.get_all_texts() + corpus.get_all_pairs()
tokens = [sentence.split() for sentence in corpus_texts]

print(tokens[:3])


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : KakaoBrain
    Repository : https://github.com/kakaobrain/KorNLUDatasets
    References :
        - Ham, J., Choe, Y. J., Park, K., Choi, I., & Soh, H. (2020). KorNLI and KorSTS: New Benchmark
           Datasets for Korean Natural Language Understanding. arXiv preprint arXiv:2004.03289.
           (https://arxiv.org/abs/2004.03289)

    This is the dataset repository for our paper
    "KorNLI and KorSTS: New Benchmark Datasets for Korean Natural Language Understanding."
    (https://arxiv.org/abs/2004.03289)
    We introduce KorNLI and KorSTS, which are NLI and STS datasets in Korean.

    # License
    Creative Commons Attribution-ShareAlike license (CC BY-SA 4.0)
    Details in https://creativecommons.org/licenses

* fastText 모델 실습

In [ ]:
from gensim.models import FastText

fastText = FastText(
    sentences = tokens,
    vector_size = 128,
    window = 5,
    min_count = 1,
    sg = 1,
    epochs = 3,
    min_n = 2,
    max_n = 6
)

# fastText.save("../models/fastText.model")
# fastText = FastText.load("../models/fastText.model")

* fastText OOV 처리

In [ ]:
oov_token = "사랑해요"
oov_vector = fastText.wv[oov_token]

print(oov_token in fastText.wv.index_to_key)
print(fastText.wv.most_similar(oov_vector, topn = 5))

False
[('사랑할까', 0.9490817189216614), ('사랑했어요', 0.948072612285614), ('사랑해요!', 0.9472817778587341), ('사랑해!', 0.9323869943618774), ('사랑했어!', 0.9170114398002625)]


# 6.5 순환 신경망

* 순환 신경망 클래스

```
rnn = torch.nn.RNN(
  input_size,
  hidden_size,
  num_layers = 1,
  nonlinaerity = "tanh",
  bias = False,
  batch_first = True,
  dropout = 0,
  bidirectional = False
)
```

* 양방향 다층 신경망

In [ ]:
import torch
from torch import nn

input_size = 128
output_size = 256
num_layers = 3
bidirectional = True

model = nn.RNN(
    input_size = input_size,
    hidden_size = output_size,
    num_layers = num_layers,
    nonlinearity = "tanh",
    batch_first = True,
    bidirectional = bidirectional
)

batch_size = 4
sequence_len = 6

inputs = torch.randn(batch_size, sequence_len, input_size)
h_0 = torch.rand(num_layers * (int(bidirectional) + 1), batch_size, output_size)

outputs, hidden = model(inputs, h_0)
print(outputs.shape)
print(hidden.shape)

torch.Size([4, 6, 512])
torch.Size([6, 4, 256])


* 장단기 메모리 클래스

```
lstm = torch.nn.LSTM(
  input_size,
  hidden_size,
  num_layers = 1,
  bias = False,
  batch_first = True,
  dropout = 0,
  bidirectional = False,
  proj_size = 0
)
```

* 양방향 다층 장단기 메모리

In [ ]:
import torch
from torch import nn

input_size = 128
output_size = 256
num_layers = 3
bidirectional = True
proj_size = 64

model = nn.LSTM(
    input_size = input_size,
    hidden_size = output_size,
    num_layers = num_layers,
    batch_first = True,
    bidirectional = bidirectional,
    proj_size = proj_size
)

batch_size = 4
sequence_len = 6

inputs = torch.rand(batch_size, sequence_len, input_size)
h_0 = torch.rand(
    num_layers * (int(bidirectional) + 1),
    batch_size,
    proj_size if proj_size > 0 else output_size
)
c_0 = torch.rand(num_layers * (int(bidirectional) + 1), batch_size, output_size)

outputs, (h_n, c_n) = model(inputs, (h_0, c_0))

print(outputs.shape)
print(h_n.shape)
print(c_n.shape)

torch.Size([4, 6, 128])
torch.Size([6, 4, 64])
torch.Size([6, 4, 256])


/usr/local/lib/python3.13/dist-packages/torch/nn/modules/rnn.py:1164: UserWarning: LSTM with projections is not supported with oneDNN. Using default implementation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/RNN.cpp:1512.)
  result = _VF.lstm(


* 문장 분류 모델

In [ ]:
from torch import nn

class SentenceClassifier(nn.Module):
  def __init__(
      self,
      n_vocab,
      hidden_dim,
      embedding_dim,
      n_layers,
      dropout = 0.5,
      bidirectional = True,
      model_type = "lstm"
  ):
      super().__init__()
      self.embedding = nn.Embedding(
          num_embeddings=n_vocab,
          embedding_dim=embedding_dim,
          padding_idx=0
      )

      if model_type == "rnn":
        self.model = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            bidirectional=bidirectional,
            dropout=dropout,
            batch_first=True
        )
      elif model_type == "lstm":
        self.model = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            bidirectional=bidirectional,
            dropout=dropout,
            batch_first=True
        )

      if bidirectional:
        self.classifier = nn.Linear(hidden_dim * 2, 1)
      else:
        self.classifier = nn.Linear(hidden_dim, 1)
      self.dropout = nn.Dropout(dropout)

  def forward(self, inputs):
      embeddings = self.embedding(inputs)
      output, _ = self.model(embeddings)
      last_output = output[:, -1, :]
      last_output = self.dropout(last_output)
      logits = self.classifier(last_output)
      return logits

* 데이터세트 불러오기

In [8]:
!pip install Korpora konlpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 78.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [konlpy]


In [9]:
import pandas as pd
from Korpora import Korpora
corpus = Korpora.load("nsmc")
corpus_df = pd.DataFrame(corpus.test)

train = corpus_df.sample(frac = 0.9, random_state = 42)
test = corpus_df.drop(train.index)

print(train.head(5).to_markdown())
print("Training Data Size:", len(train))
print("Testing Data Size:", len(test))


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/



[nsmc] download ratings_train.txt: 14.6MB [00:00, 42.5MB/s]                            
[nsmc] download ratings_test.txt: 4.90MB [00:00, 18.2MB/s]                           


|       | text                                                                                     |   label |
|------:|:-----------------------------------------------------------------------------------------|--------:|
| 33553 | 모든 편견을 날려 버리는 가슴 따뜻한 영화. 로버트 드 니로, 필립 세이모어 호프만 영원하라. |       1 |
|  9427 | 무한 리메이크의 소재. 감독의 역량은 항상 그 자리에...                                    |       0 |
|   199 | 신날 것 없는 애니.                                                                       |       0 |
| 12447 | 잔잔 격동                                                                                |       1 |
| 39489 | 오랜만에 찾은 주말의 명화의 보석                                                         |       1 |
Training Data Size: 45000
Testing Data Size: 5000


* 데이터 토큰화 및 단어 사전 구축

In [11]:
from konlpy.tag import Okt
from collections import Counter

def build_vocab(corpus, n_vocab, special_tokens):
  counter = Counter()
  for tokens in corpus:
    counter.update(tokens)
  vocab = special_tokens
  for token, count in counter.most_common(n_vocab):
    vocab.append(token)
  return vocab
tokenizer = Okt()
train_tokens = [tokenizer.morphs(review) for review in train.text]
test_tokens = [tokenizer.morphs(review) for review in test.text]

vocab = build_vocab(corpus = train_tokens, n_vocab = 5000, special_tokens = ["<pad>", "<unk>"])
token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for idx, token in enumerate(vocab)}

print(vocab[:10])
print(len(vocab))

['<pad>', '<unk>', '.', '이', '영화', '의', '..', '가', '에', '...']
5002


* 정수 인코딩 및 패딩

In [12]:
import numpy as np

def pad_sequences(sequences, max_length, pad_value):
  result = list()
  for sequence in sequences:
    sequence = sequence[:max_length]
    pad_length = max_length - len(sequence)
    padded_sequence = sequence + [pad_value] * pad_length
    result.append(padded_sequence)
  return np.asarray(result)

unk_id = token_to_id["<unk>"]
train_ids = [
    [token_to_id.get(token, unk_id) for token in review] for review in train_tokens
]
test_ids = [
    [token_to_id.get(token, unk_id) for token in review] for review in test_tokens
]

max_length = 32
pad_id = token_to_id["<pad>"]
train_ids = pad_sequences(train_ids, max_length, pad_id)
test_ids = pad_sequences(test_ids, max_length, pad_id)

print(train_ids[0])
print(test_ids[0])

[ 223 1716   10 4036 2095  193  755    4    2 2330 1031  220   26   13
 4839    1    1    1    2    0    0    0    0    0    0    0    0    0
    0    0    0    0]
[3307    5 1997  456    8    1 1013 3906    5    1    1   13  223   51
    3    1 4684    6    0    0    0    0    0    0    0    0    0    0
    0    0    0    0]


* 데이터로더 적용

In [14]:
import torch
from torch.utils.data import TensorDataset, DataLoader

train_ids = torch.tensor(train_ids)
test_ids = torch.tensor(test_ids)

train_labels = torch.tensor(train.label.values, dtype = torch.float32)
test_labels = torch.tensor(test.label.values, dtype = torch.float32)

train_dataset = TensorDataset(train_ids, train_labels)
test_dataset = TensorDataset(test_ids, test_labels)

train_loader = DataLoader(train_dataset, batch_size = 16, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 16, shuffle = False)

/tmp/ipykernel_445/4032982136.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_ids = torch.tensor(train_ids)
/tmp/ipykernel_445/4032982136.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_ids = torch.tensor(test_ids)


* 손실 함수와 최적화 함수 정의

In [15]:
from torch import optim

n_vocab = len(token_to_id)
hidden_dim = 64
embedding_dim = 128
n_layers = 2

device = "cuda" if torch.cuda.is_available() else "cpu"
classifier = SentenceClassifier(
    n_vocab = n_vocab, hidden_dim = hidden_dim, embedding_dim = embedding_dim, n_layers = n_layers
).to(device)
criterion = nn.BCEWithLogitsLoss().to(device)
optimizer = optim.RMSprop(classifier.parameters(), lr = 0.001)

* 모델 학습 및 테스트

In [18]:
def train(model, datasets, criterion, optimizer, device, interval):
  model.train()
  losses = list()

  for step, (input_ids, labels) in enumerate(datasets):
    input_ids = input_ids.to(device)
    labels = labels.to(device).unsqueeze(1)

    logits = model(input_ids)
    loss = criterion(logits, labels)
    losses.append(loss.item())

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % interval == 0:
      print(f"Train Loss {step}: {np.mean(losses)}")

def test(model, datasets, criterion, device):
  model.eval()
  losses = list()
  corrects = list()

  for step, (input_ids, labels) in enumerate(datasets):
    input_ids = input_ids.to(device)
    labels = labels.to(device).unsqueeze(1)

    logits = model(input_ids)
    loss = criterion(logits, labels)
    losses.append(loss.item())
    yhat = torch.sigmoid(logits)>.5
    corrects.extend(
        torch.eq(yhat, labels).cpu().tolist()
    )

  print(f"Val Loss: {np.mean(losses)}, Val Accuracy : {np.mean(corrects)}")

epochs = 5
interval = 500
for epoch in range(epochs):
  train(classifier, train_loader, criterion, optimizer, device, interval)
  test(classifier, test_loader, criterion, device)

Train Loss 0: 0.6969212293624878
Train Loss 500: 0.693628186475255
Train Loss 1000: 0.6898161382465572
Train Loss 1500: 0.68043027569023
Train Loss 2000: 0.6745200373869071
Train Loss 2500: 0.669668391388638
Val Loss: 0.6382186481366142, Val Accuracy : 0.6592
Train Loss 0: 0.5469891428947449
Train Loss 500: 0.6307218758883828
Train Loss 1000: 0.6288111215526169
Train Loss 1500: 0.6208558610524438
Train Loss 2000: 0.6150281372396783
Train Loss 2500: 0.6070721423277994
Val Loss: 0.5300806667477178, Val Accuracy : 0.7378
Train Loss 0: 0.40288257598876953
Train Loss 500: 0.5189944741552224
Train Loss 1000: 0.5095025927542925
Train Loss 1500: 0.5007506240057517
Train Loss 2000: 0.49117437894197774
Train Loss 2500: 0.48379488170194607
Val Loss: 0.4477216855119973, Val Accuracy : 0.7924
Train Loss 0: 0.4744287133216858
Train Loss 500: 0.42626192212521197
Train Loss 1000: 0.41924857194875026
Train Loss 1500: 0.41575352824742917
Train Loss 2000: 0.41363407747781855
Train Loss 2500: 0.4107708960

* 학습된 모델로부터 임베딩 추출

In [19]:
token_to_embedding = dict()
embedding_matrix = classifier.embedding.weight.detach().cpu().numpy()

for word, emb in zip(vocab, embedding_matrix):
  token_to_embedding[word] = emb

token = vocab[1000]
print(token, token_to_embedding[token])

보고싶다 [ 2.5450542   0.17277332 -0.61634904  0.13030377 -0.47479647 -0.8804234
 -0.8965424  -2.899826   -1.1255243  -1.6302713   1.9689273  -1.6656598
  0.0181771   0.3235328   1.5336889  -1.079683    0.9745331   0.77856535
  1.7089785  -0.12072967 -0.14499448  0.66541445  1.2720046  -1.1561173
  0.3776504  -1.8736198   0.641503    1.2188529   1.3098791  -1.8106035
 -0.04722415 -0.68152106  1.4104754   0.7225465   1.7175785   0.6049325
  2.2140183   0.08821158 -0.45430708  0.49092636  0.84572875 -1.046856
 -2.8529222   0.93443763 -0.8253849   0.65253526 -0.29078558  1.2294568
  1.287356    0.35182312  0.20100802  0.31454277  1.4437398   1.3005538
 -1.7578827   0.7155115  -1.4650768  -1.1139562  -0.46665522 -0.37490278
  0.44723913  1.7167822  -0.1816394  -0.6086929   1.0798148   1.2435127
 -0.56825364  1.3989431  -1.391852    1.1346205   0.35139665 -0.08257531
 -0.7509929  -0.12624383  0.8691256   0.98412764 -0.35263598 -0.5491436
  1.1365166  -0.71090984  1.207936   -0.6826745  -0.86395

* 사전 학습된 모델로 임베딩 계층 초기화

In [21]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 81.6 MB/s  0:00:00


In [25]:
from gensim.models import Word2Vec

# 파일이 없으므로 현재 메모리에 있는 train_tokens로 바로 학습
word2vec = Word2Vec(
    sentences=train_tokens,
    vector_size=embedding_dim,
    window=5,
    min_count=1,
    sg=1
)

init_embeddings = np.zeros((n_vocab, embedding_dim))

for index, token in id_to_token.items():
  if token not in ["<pad>", "<unk>"] and token in word2vec.wv:
    init_embeddings[index] = word2vec.wv[token]  # vc -> wv 오타 수정

embedding_layer = nn.Embedding.from_pretrained(
    torch.tensor(init_embeddings, dtype=torch.float32)
)

* 사전 학습된 임베딩 계층 적용

In [27]:
class SentenceClassifier(nn.Module):
  def __init__(
      self,
      n_vocab,
      hidden_dim,
      embedding_dim,
      n_layers,
      dropout=0.5,
      bidirectional=True,
      model_type="lstm",
      pretrained_embedding=None
  ):
    super().__init__()

    if pretrained_embedding is not None:
      self.embedding = nn.Embedding.from_pretrained(
          torch.tensor(pretrained_embedding, dtype=torch.float32)
      )
    else:
      self.embedding = nn.Embedding(
          num_embeddings=n_vocab,
          embedding_dim=embedding_dim,
          padding_idx=0
      )

    if model_type == "rnn":
      self.model = nn.RNN(
          input_size=embedding_dim,
          hidden_size=hidden_dim,
          num_layers=n_layers,
          bidirectional=bidirectional,
          dropout=dropout,
          batch_first=True
      )
    elif model_type == "lstm":
      self.model = nn.LSTM(
          input_size=embedding_dim,
          hidden_size=hidden_dim,
          num_layers=n_layers,
          bidirectional=bidirectional,
          dropout=dropout,
          batch_first=True
      )

    if bidirectional:
      self.classifier = nn.Linear(hidden_dim * 2, 1)
    else:
      self.classifier = nn.Linear(hidden_dim, 1)
    self.dropout = nn.Dropout(dropout)

  def forward(self, inputs):
    embeddings = self.embedding(inputs)
    output, _ = self.model(embeddings)
    last_output = output[:, -1, :]
    last_output = self.dropout(last_output)
    logits = self.classifier(last_output)
    return logits

* 사전 학습된 임베딩을 사용한 모델 학습

In [29]:
classifier = SentenceClassifier(
    n_vocab = n_vocab, hidden_dim = hidden_dim, embedding_dim = embedding_dim,
    n_layers = n_layers, pretrained_embedding = init_embeddings
).to(device)
criterion = nn.BCEWithLogitsLoss().to(device)
optimizer = optim.RMSprop(classifier.parameters(), lr = 0.001)

epoch = 5
interval = 500

for epoch in range(epochs):
  train(classifier, train_loader, criterion, optimizer, device, interval)
  test(classifier, test_loader, criterion, device)

Train Loss 0: 0.6828538179397583
Train Loss 500: 0.6876923546581687
Train Loss 1000: 0.6695661283694543
Train Loss 1500: 0.6277024189108297
Train Loss 2000: 0.5979600086294371
Train Loss 2500: 0.5764286034777373
Val Loss: 0.4462814926148984, Val Accuracy : 0.7966
Train Loss 0: 0.5791267156600952
Train Loss 500: 0.4707807000287755
Train Loss 1000: 0.47479303018911023
Train Loss 1500: 0.4715510137195511
Train Loss 2000: 0.46627643225953797
Train Loss 2500: 0.46328187515227043
Val Loss: 0.4301010365493762, Val Accuracy : 0.8006
Train Loss 0: 0.4772266745567322
Train Loss 500: 0.44791793085619835
Train Loss 1000: 0.4456735273281654
Train Loss 1500: 0.4462164622160056
Train Loss 2000: 0.4521646477777263
Train Loss 2500: 0.4512519762485993
Val Loss: 0.44133621425674363, Val Accuracy : 0.7804
Train Loss 0: 0.6005944609642029
Train Loss 500: 0.4294672867435657
Train Loss 1000: 0.43662590695011033
Train Loss 1500: 0.4330525472809838
Train Loss 2000: 0.4285468947792101
Train Loss 2500: 0.4280672